In [60]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

fraud_df = pd.read_csv("../data/raw/fraud_data.csv")

fraud_df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


In [61]:
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])

In [62]:
fraud_df['time_since_signup'] = (
    fraud_df['purchase_time'] - fraud_df['signup_time']
).dt.total_seconds() / 3600

fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour

fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.day_name()

fraud_df['transaction_velocity'] = (
    fraud_df.groupby('device_id')['device_id']
    .transform('count')
)

In [63]:
categorical_features = [
    'source',
    'browser',
    'sex',
    'day_of_week'
]

fraud_encoded = pd.get_dummies(
    fraud_df,
    columns=categorical_features,
    drop_first=True
)

fraud_encoded.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,age,ip_address,class,time_since_signup,hour_of_day,...,browser_IE,browser_Opera,browser_Safari,sex_M,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,39,7.327584e+08,0,1251.856111,2,...,False,False,False,True,False,True,False,False,False,False
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,53,3.503114e+08,0,4.984444,1,...,False,False,False,False,True,False,False,False,False,False
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,53,2.621474e+09,1,0.000278,18,...,False,True,False,True,False,False,False,True,False,False
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,41,3.840542e+09,0,136.690278,13,...,False,False,True,True,True,False,False,False,False,False
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,45,4.155831e+08,0,1211.516944,18,...,False,False,True,True,False,False,False,False,False,True


In [64]:
from sklearn.preprocessing import StandardScaler

numerical_features = [
    'purchase_value',
    'age',
    'time_since_signup',
    'hour_of_day',
    'transaction_velocity'
]

scaler = StandardScaler()

fraud_encoded[numerical_features] = scaler.fit_transform(
    fraud_encoded[numerical_features]
)

fraud_encoded[numerical_features].head()

,purchase_value,age,time_since_signup,hour_of_day,transaction_velocity
0,-0.160204,0.679914,-0.136057,-1.377455,-0.261514
1,-1.142592,2.304476,-1.571877,-1.522122,-0.261514
2,-1.197169,2.304476,-1.577617,0.937208,3.941861
3,0.385567,0.911994,-1.420213,0.213876,-0.261514
4,0.112681,1.376155,-0.182509,0.937208,-0.261514


In [65]:
fraud_encoded.shape

(151112, 24)

## Data Transformation

### One-Hot Encoding

Categorical variables were transformed using one-hot encoding:

- source
- browser
- sex
- day_of_week

### Feature Scaling

Numerical variables were standardized using StandardScaler:

- purchase_value
- age
- time_since_signup
- hour_of_day
- transaction_velocity

Scaling ensures that numerical features have comparable ranges and improves machine learning model performance.

In [66]:
X = fraud_encoded.drop(
    columns=[
        'class',
        'device_id'
    ]
)

y = fraud_encoded['class']

print(X.shape)

(151112, 22)


In [67]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Set:", X_train.shape)
print("Test Set:", X_test.shape)

Training Set: (120889, 22)
Test Set: (30223, 22)


In [ ]:
print("Before SMOTE")

print(y_train.value_counts())

Before SMOTE
class
0    109568
1     11321
Name: count, dtype: int64


In [ ]:
X = X.select_dtypes(exclude=['datetime64[ns]'])

print(X.dtypes)
print(X.shape)

user_id                    int64
purchase_value           float64
age                      float64
ip_address               float64
time_since_signup        float64
hour_of_day              float64
transaction_velocity     float64
source_Direct               bool
source_SEO                  bool
browser_FireFox             bool
browser_IE                  bool
browser_Opera               bool
browser_Safari              bool
sex_M                       bool
day_of_week_Monday          bool
day_of_week_Saturday        bool
day_of_week_Sunday          bool
day_of_week_Thursday        bool
day_of_week_Tuesday         bool
day_of_week_Wednesday       bool
dtype: object
(151112, 20)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("After SMOTE")
print(y_train_smote.value_counts())

After SMOTE
class
0    109568
1    109568
Name: count, dtype: int64


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("After SMOTE")

print(y_train_smote.value_counts())

After SMOTE
class
0    109568
1    109568
Name: count, dtype: int64


In [ ]:
X.columns

Index(['user_id', 'purchase_value', 'device_id', 'age', 'ip_address',
       'time_since_signup', 'hour_of_day', 'transaction_velocity',
       'source_Direct', 'source_SEO', 'browser_FireFox', 'browser_IE',
       'browser_Opera', 'browser_Safari', 'sex_M', 'day_of_week_Monday',
       'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday',
       'day_of_week_Tuesday', 'day_of_week_Wednesday'],
      dtype='object')

In [ ]:
X.dtypes

user_id                    int64
purchase_value           float64
device_id                 object
age                      float64
ip_address               float64
time_since_signup        float64
hour_of_day              float64
transaction_velocity     float64
source_Direct               bool
source_SEO                  bool
browser_FireFox             bool
browser_IE                  bool
browser_Opera               bool
browser_Safari              bool
sex_M                       bool
day_of_week_Monday          bool
day_of_week_Saturday        bool
day_of_week_Sunday          bool
day_of_week_Thursday        bool
day_of_week_Tuesday         bool
day_of_week_Wednesday       bool
dtype: object

## Class Imbalance Handling with SMOTE

The training dataset exhibited significant class imbalance:

### Before SMOTE
- Non-Fraud (Class 0): 109,568
- Fraud (Class 1): 11,321

To address this issue, Synthetic Minority Oversampling Technique (SMOTE) was applied to the training data only.

### After SMOTE
- Non-Fraud (Class 0): 109,568
- Fraud (Class 1): 109,568

SMOTE generated synthetic fraud samples to balance the classes while preserving the original test set for unbiased model evaluation.

## Class Imbalance Handling

The dataset exhibited significant class imbalance, with fraudulent transactions representing a minority of observations.

To address this issue:

- The dataset was first split into training and testing sets.
- SMOTE (Synthetic Minority Oversampling Technique) was applied only to the training set.
- The test set was left untouched to prevent data leakage.

This approach improves the model's ability to learn fraud patterns while maintaining a realistic evaluation process.